# Comparative Analysis of CNN and Transfer Learning for Flower Image Classification

This notebook investigates the impact of transfer learning on image classification performance under limited data conditions.

We compare:
- A custom CNN trained from scratch
- A pretrained DenseNet121 model fine-tuned on the target dataset

The objective is to analyze generalization behavior rather than only final accuracy.

# I- Data preparation 

## 1. Data load 

In [ ]:
# Imports and Configuration

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow.keras import callbacks
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data import AUTOTUNE

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.prepare_data import extract_zip
from src.models import build_cnn_v1, build_cnn_v2, build_densenet
from src.evaluate import plot_learning_curves, evaluate_model

DATA_DIR = PROJECT_ROOT / "data" / "raw"
ZIP_DIR = PROJECT_ROOT / "src" / "dataset.zip"

extract_zip(ZIP_DIR, DATA_DIR)

In [3]:
# Load datasets
train_ds = image_dataset_from_directory(
    DATA_DIR/"jpg",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(128, 128),
    batch_size=16
)

val_ds = image_dataset_from_directory(
    DATA_DIR/"jpg",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(128, 128),
    batch_size=16
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes :", class_names)



Found 320 files belonging to 4 classes.
Using 256 files for training.
Found 320 files belonging to 4 classes.
Using 64 files for validation.
Classes : ['0', '2', '4', '9']


In [4]:
# Optimize input pipeline
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

## 2. Data augmentation 

In [5]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


# II-Baseline CNN

## 1. Architecture and Training




We design a VGG-style convolutional network with increasing channel depth.

The architecture progressively increases feature dimensionality
(64 → 128 → 256 filters) while reducing spatial resolution through max pooling.

Global average pooling is used instead of flattening to reduce parameter count
and mitigate overfitting in a low-data regime.

The model was designed to satisfy the constraint of having between 1 and 3 million parameters.  
However, the experimental results suggest that a more compact network achieves better validation performance.

This observation highlights that, in a limited data regime, model capacity must be carefully controlled. An overly complex network tends to overfit quickly, even when regularization techniques such as dropout and batch normalization are applied.


In [ ]:
model = build_cnn_v1(num_classes=num_classes)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stop]
)

## 2. Model Evaluation and Performance Analysis


In [ ]:
plot_learning_curves(history, title_prefix="CNN v1")
evaluate_model(model, val_ds, class_names)

The learning curves reveal a significant gap between training and validation performance, indicating overfitting.
While training accuracy reaches above 80%, validation accuracy remains around 45%, with highly unstable validation loss.

The confusion matrix highlights a strong class imbalance in predictions, with the model over-predicting class "2" and completely failing to recognize class "9".
These results suggest that the model capacity may be too high relative to the dataset size, leading to poor generalization despite regularization techniques.

To improve the results, several strategies could be explored. First, stronger regularization could be applied by reducing model capacity (e.g., decreasing the size of the dense layer), increasing dropout, or lowering the learning rate. Second, more aggressive data augmentation could help improve generalization. Finally, collecting more training data or using transfer learning with a pretrained model could significantly enhance performance and class balance.

## 3. Improvement 

Based on the previous evaluation, we introduce a revised architecture aimed at reducing overfitting and improving generalization under limited data conditions.

In [ ]:
model2 = build_cnn_v2(num_classes=num_classes, learning_rate=1e-4)

model2.summary()

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

history2 = model2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
plot_learning_curves(history2, title_prefix="CNN v2")
evaluate_model(model2, val_ds, class_names)

Despite reducing model capacity and strengthening regularization, the revised architecture does not significantly improve validation performance. The gap between training and validation accuracy remains substantial, and several classes (notably "4" and "9") are still poorly recognized.

These results suggest that the main limitation may not lie solely in model capacity, but rather in the extremely limited dataset size. In such a low-data regime, training a convolutional network from scratch appears unstable and prone to overfitting, even with architectural adjustments and optimization strategies.

This motivates the exploration of transfer learning, where pretrained representations may provide more robust and stable feature extraction under strong data constraints.

# II- Transfer Learning with DenseNet121

## 1. Architecture and Training

Given the overfitting observed when training a CNN from scratch, we now explore a transfer learning approach. 

We use a DenseNet121 model pretrained on ImageNet as a fixed feature extractor, and train only a small classification head on our dataset. This allows us to leverage rich pretrained representations while limiting the number of trainable parameters.

In [ ]:
densenet_model = build_densenet(num_classes=num_classes)

densenet_model.summary()

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_densenet = densenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stop]
)

## 2. Model Evaluation and Performance Analysis

In [ ]:
plot_learning_curves(history_densenet, title_prefix="DenseNet121")
evaluate_model(densenet_model, val_ds, class_names)

The DenseNet121-based model achieves strong and stable performance, with high validation accuracy (98%) and consistently low validation loss. The learning curves indicate rapid convergence and no significant generalization gap.

The confusion matrix confirms that all classes are correctly identified with high precision and recall, suggesting that the pretrained feature representations are well suited to this classification task.

These results demonstrate the effectiveness of transfer learning in a low-data setting, where leveraging pretrained visual features allows the model to achieve robust and reliable performance.


# III-Comparison Between Custom CNN and DenseNet121


## Which model is better?

The three experiments reveal a clear difference in behavior between training a CNN from scratch and using transfer learning.

The custom CNN models, despite architectural adjustments and stronger regularization, consistently exhibited overfitting. A significant gap between training and validation performance remained, and some classes were poorly recognized. These results suggest that learning visual features from scratch with a limited dataset leads to unstable generalization.

In contrast, the DenseNet121-based model achieved high validation accuracy (98%) with stable learning dynamics and strong performance across all classes. The pretrained backbone provides robust visual representations that generalize well even with limited training samples.

This comparison highlights a key insight: under strong data constraints, transfer learning is significantly more effective than training a deep convolutional architecture from scratch. Pretrained representations drastically reduce the need for large amounts of task-specific data while improving stability and class balance.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.densenet import preprocess_input

# Detect project root
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load test image
IMG_PATH = PROJECT_ROOT / "data" / "assets" / "test.jpg"
img = image.load_img(IMG_PATH, target_size=(128, 128))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

# Predict with custom CNN (v2)
pred_cnn = model2.predict(img_array)
pred_class_cnn = class_names[np.argmax(pred_cnn)]

# Predict with DenseNet121 (uses its own preprocessing internally)
pred_densenet = densenet_model.predict(img_array)
pred_class_densenet = class_names[np.argmax(pred_densenet)]

print("Custom CNN Prediction  :", pred_class_cnn)
print("DenseNet121 Prediction :", pred_class_densenet)

The CNN trained from scratch misclassified the test image, whereas the DenseNet121 model predicted the correct class. This confirms the previous analysis: the CNN overfitted due to the limited size of the training dataset relative to its model capacity (over one million parameters). In contrast, transfer learning provided more robust feature representations and better generalization on unseen data.